## DSAI4205 Big Data Analytics
## Assignment 2 (10%), Term 2, 2025-2026

#### **Instructions for the assignment**:
* Follow the provided instructions to solve the questions.
* Please save the report and named as DSAI4205Assignment2_{your student_id}.ipynb (e.g., if student_id is 1234567, then the file's name is DSAI4205Assignment2_1234567.ipynb).

* Submit the notebook file to the blackboard. The submission due date is **<font color='red'>24 Mar 2026 23:55</font>**

<font color='red'>**You should complete the assignment on your own. You must uphold academic integrity and academic honesty in the assignment. You can be subject to disciplinary action if proven to have acted against academic integrity and academic honesty.**</font>

### Question 1 – Car Purchase History Analysis (20%)

Write a module to read the purchase history file (i.e., car_sales.txt); then count and display the most frequent **FIVE** car brands that being sold in the log. The results **MUST** contain the **“top-5 frequent Car Brands and their sales count”**. You can return the results in any format (e.g., list, dict), not necessarily in dataframe. E.g., (Car Brand, count)

The purchase history file that we use for this assignment is the car sales file. The file looks something like the following: `Ford - - 2015 SUV Auto Black 26`

Each part of this file is described below.
- `Ford`: This is the car brand of one sales record, which means that for this sales record, the car that being sold is a Ford car. **(This is what you need to extract!)**
- `-`: The "hyphen" in the output indicates that the requested piece of information (user identity) is not available.
- `-`: The "hyphen" in the output indicates that the requested piece of information (user DOB) is not available.
- `2015` : The time that the car was being sold.
- `SUV` : This is the car type of the sale record, we have "SUV", "Passenger", "Hardtop" and so on.
- `Auto` : This means the car is in "Auto mode" or in "Manual mode".
- `Black` : This is the color of the car.
- `26` : This is the sales price of the car (26 means 26,000$).


In general, your module will contain (but not limited to) the following parts:

1. A component that reads the log file
1. A function that extracts the car brand (e.g., Ford, Chevrolet, … ) from the file (**Hints**: the car brands are **the starting string to the first “-” of each row**. You may use **slicing** to extract the substring from text; and use map() to apply slicing on each row of the text file)
1. A component that counts the car sales amount and sort them descendingly.


In [ ]:
# Please provide your answer here

def read_sales_file(filepath):
    """Read the car sales log file and return a list of non-empty lines."""
    with open(filepath, "r") as f:
        lines = f.readlines()
    # Strip whitespace and keep only non-empty lines
    lines = [line.strip() for line in lines if line.strip()]
    return lines


def extract_brand(line: str) -> str:
    """
    Extract the car brand from a single sales record line.
    The brand is everything before the first occurrence of ' - '.
    Uses string slicing via .index() to locate the delimiter.
    """
    delimiter = " - "
    end_index = line.index(delimiter)  # position of the first ' - '
    return line[:end_index].strip()    # slice from start up to delimiter

def extract_all_brands(lines: list[str]) -> list[str]:
    """Apply extract_brand to every line using map()."""
    return list(map(extract_brand, lines))


def count_brands(brands):
    """
    Count how many times each brand appears using a plain dictionary.
    """
    brand_counts = {}
    for brand in brands:
        if brand in brand_counts:
            brand_counts[brand] += 1
        else:
            brand_counts[brand] = 1
    return brand_counts


def top_n_brands(brand_counts, n=5):
    """
    Sort the brand counts dictionary descending by count,
    and return the top-n results as a list of (brand, count) tuples.
    """
    # Convert dict to list of (brand, count) tuples
    brand_list = list(brand_counts.items())

    # Sort descending by count using sorted() with a key function
    for i in range(len(brand_list)):
        for j in range(len(brand_list) - 1 - i):
            if brand_list[j][1] < brand_list[j + 1][1]:
                brand_list[j], brand_list[j + 1] = brand_list[j + 1], brand_list[j]

    # Return only the top n
    return brand_list[:n]

if __name__ == "__main__":

    lines = read_sales_file(r"/content/drive/MyDrive/A2/car_sales.txt") #since my local kernel has java version conflict,I use colab platform to do it.
    print(f"Total sales records read: {len(lines)}\n")

    # Step 2: Extract brands using map()
    brands = extract_all_brands(lines)

    # Step 3: Count brands manually with a dictionary
    brand_counts = count_brands(brands)

    # Step 4: Sort descending and get top 5
    top5 = top_n_brands(brand_counts, n=5)

    # Display results
    print("=" * 42)
    print("  Top-5 Most Frequent Car Brands Sold")
    print("=" * 42)
    print(f"  {'Car Brand':<18} {'Sales Count':>10}")
    print("-" * 42)
    for rank, (brand, count) in enumerate(top5, start=1):
        print(f"  {rank}. {brand:<15} {count:>10}")
    print("=" * 42)

    # Also return as a list of tuples for programmatic use



Total sales records read: 2443

  Top-5 Most Frequent Car Brands Sold
  Car Brand          Sales Count
------------------------------------------
  1. Chevrolet              177
  2. Dodge                  173
  3. Ford                   167
  4. Mitsubishi             146
  5. Volkswagen             131


### Question 2 - Big Data Analysis for Student Plagiarism Detection (80%)

a)	Construct a Student Table as a PySpark DataFrame containing relevant student information as below. (The content is provided in the file student_data.txt, which can be found on Blackboard) (10%)

In [ ]:
!pip install pyspark==3.5.3

In [ ]:
!java -version

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED -Djdk.attach.allowAttachSelf=true
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!pyspark --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.3
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 17.0.17
Branch HEAD
Compiled by user haejoon.lee on 2024-09-09T05:20:05Z
Revision 32232e9ed33bb16b93ad58cfde8b82e0f07c0970
Url https://github.com/apache/spark
Type --help for more information.


<font color="red">Task 2.1.1: Construct a Student Table as a PySpark DataFrame.</font>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ArrayType
from pyspark.sql.functions import udf, col, collect_list, struct, lit
import os, sys


# Define schema
schema = StructType([
    StructField("StudentID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Grade", StringType(), True),
    StructField("Year of Study", IntegerType(), True),
    StructField("Major", StringType(), True),
    StructField("Homework", StringType(), True),
    StructField("Past GPA", FloatType(), True),
])

# Define 10 rows of data with mixed homework similarities
data = [
    ("12aB34", "Alice", 20, "F", "B+", 2, "CS", "The sky is blue. I enjoy reading books. Cats are curious.", 3.5),
    ("44rT32", "Daniel", 20, "M", "B", 2, "CS", "Cats are curious. The sky is blue. I enjoy reading books.", 3.4),
    ("05Xx21", "Nora", 21, "F", "A", 3, "CS", "Algorithms are efficient. Data structures organize data. Python is popular.", 3.6),
    ("98Zx56", "Brian", 21, "M", "A-", 3, "Lit", "I enjoy reading books. Cats are curious. The sky is blue.", 3.7),
    ("75qQ12", "Faisal", 23, "M", "C+", 5, "Lit", "Shakespeare wrote plays. Poetry uses metaphors. Literature reflects society.", 2.9),
    ("67Lm89", "Carla", 22, "F", "A", 4, "Bio", "Photosynthesis needs sunlight. Leaves are green. Plants grow in soil.", 3.9),
    ("88Hg45", "Grace", 21, "F", "C", 3, "Bio", "Leaves are green. Plants grow in soil. Photosynthesis needs sunlight.", 1.9),
    ("31Oo76", "Emily", 19, "F", "A-", 1, "Hist", "The past shapes the future. Wars affect nations. People write history.", 3.8),
    ("22Ws90", "Henry", 20, "M", "B-", 2, "Hist", "Trade routes shaped empires. The Renaissance sparked ideas. Monarchs ruled nations.", 3.2),
    ("59Kk38", "Jack", 22, "M", "B", 4, "EnvSci", "Climate change affects weather. Ecosystems are fragile. We must reduce emissions.", 3.3),
]

# Pass Python path via Spark config — more reliable than os.environ on Windows (due to version conflict between Java sdk, python and pyspark in my local laptop)
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()



df = spark.createDataFrame(data, schema=schema)
df.show(truncate=False)


+---------+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+
|StudentID|Name  |Age|Gender|Grade|Year of Study|Major |Homework                                                                           |Past GPA|
+---------+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+
|12aB34   |Alice |20 |F     |B+   |2            |CS    |The sky is blue. I enjoy reading books. Cats are curious.                          |3.5     |
|44rT32   |Daniel|20 |M     |B    |2            |CS    |Cats are curious. The sky is blue. I enjoy reading books.                          |3.4     |
|05Xx21   |Nora  |21 |F     |A    |3            |CS    |Algorithms are efficient. Data structures organize data. Python is popular.        |3.6     |
|98Zx56   |Brian |21 |M     |A-   |3            |Lit   |I enjoy reading books. Cats are curious. The

<font color="red">Task 2.1.2: Export the DataFrame to a JSON file.</font>

In [ ]:
# Please provide your answer here


pandas_df = df.toPandas()
output_path = '/content/drive/MyDrive/A2/student_data.json'
pandas_df.to_json(output_path, orient="records", indent=2)
print("Task 2.1.2: DataFrame exported to student_data.json")

Task 2.1.2: DataFrame exported to student_data.json


b)	Compute Similarity Scores Among Students of the Same Major. (30%)
For students within the same major, compare their homework submissions and calculate a similarity score based on ROUGE-L metric.


In [ ]:
!pip install rouge-score

In [ ]:
from collections import defaultdict
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, FloatType, IntegerType
from rouge_score import rouge_scorer

# TODO 2.2.1: Tokenize Homework
def tokenize_homework(homework):
    """Split homework text into individual sentences."""
    sentences = [s.strip() for s in homework.split('.') if s.strip()]
    return sentences


# TODO 2.2.2: Define ROUGE-L similarity computation between two sentences
def compute_rouge_l(sentences_a, sentences_b):
    """
    Compare every sentence in A's homework with each sentence in B's homework
    using ROUGE-L similarity, and return the average score.
    """


    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    total_score = 0.0

    for sent_a in sentences_a:
        best_score = 0.0
        for sent_b in sentences_b:
            scores = scorer.score(sent_a, sent_b)
            fmeasure = scores['rougeL'].fmeasure
            if fmeasure > best_score:
                best_score = fmeasure
        total_score += best_score

    return round(total_score, 4)

# TODO 2.2.3: Define the udf function
# Broadcast plain dicts so workers can access them without SparkContext
broadcast_students = spark.sparkContext.broadcast(
    [row.asDict() for row in df.collect()]
)



# TODO 2.2.4: Perform comparison within the same major

def compute_similarities(name, major, homework):
    # Use the broadcasted data instead of calling df.collect() inside the UDF
    all_students_data = broadcast_students.value
    sim_names = []
    sim_scores = []
    sentences_a = tokenize_homework(homework)

    for student_b in all_students_data:
        # Ensure we don't compare a student with themselves and check for the same major
        if student_b["Name"] != name and student_b["Major"] == major:
            sentences_b = tokenize_homework(student_b["Homework"])
            score = compute_rouge_l(sentences_a, sentences_b)
            sim_names.append(student_b["Name"])
            sim_scores.append(float(score))

    return (sim_names, sim_scores)

# Only one definition of similarity_scores_udf is needed
similarity_scores_udf = udf(
    lambda name, major, homework: compute_similarities(name, major, homework),
    StructType([
        StructField("sim_names", ArrayType(StringType()), True),
        StructField("sim_scores", ArrayType(FloatType()), True),
    ])
)

result_df = df.withColumn(
    "similarity",
    similarity_scores_udf(col("Name"), col("Major"), col("Homework"))
).select(
    "Name", "Age", "Gender", "Grade",
    "Year of Study", "Major", "Homework", "Past GPA",
    col("similarity.sim_names").alias("sim_names"),
    col("similarity.sim_scores").alias("sim_scores")
)

# TODO 2.2.5: Sort by Major and StudentID, then show the results
result_df = result_df.sort(col("Major"), col("StudentID"))
# result_df = result_df.sort(col("Major"), col("StudentID")).drop("StudentID")
result_df.show(truncate=False)



+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+---------------+--------------+
|Name  |Age|Gender|Grade|Year of Study|Major |Homework                                                                           |Past GPA|sim_names      |sim_scores    |
+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+---------------+--------------+
|Carla |22 |F     |A    |4            |Bio   |Photosynthesis needs sunlight. Leaves are green. Plants grow in soil.              |3.9     |[Grace]        |[3.0]         |
|Grace |21 |F     |C    |3            |Bio   |Leaves are green. Plants grow in soil. Photosynthesis needs sunlight.              |1.9     |[Carla]        |[3.0]         |
|Nora  |21 |F     |A    |3            |CS    |Algorithms are efficient. Data structures organize data. Python is popular.        |3.6     |[Alice

c)	Identify Plagiarism Cases: (20%)

Assume that a similarity score of 3 (i.e., sim_scores = 3) indicates plagiarism between students of the same major.
Implement a DataFrame querying system to answer the following questions:
- "Who plagiarized in Major X?" (x mean a particular Major, e.g., Bio)
- "How many students plagiarized in each major?"
- Notes:
    - (15%) will be awarded for correctly implementing the query system.
    - (5%) will be awarded if the system can handle different phrasings or patterns of similar questions
    (e.g., "Who copied in Major X?", "List students who cheated in CS").


In [ ]:
#import Pyspark modules
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
import re

def answer_question(question, df):

     # TODO 2.3.1: Preparing the dataframe for answering the question
      # Build plagiarism pairs with BOTH students' majors checked
      exploded_df = (result_df.select(col("Name").alias("StudentA"),col("Major").alias("MajorA"),F.explode(F.arrays_zip(col("sim_names"), col("sim_scores"))).alias("pair")
              ).select(col("StudentA"),col("MajorA"),col("pair.sim_names").alias("StudentB"),col("pair.sim_scores").alias("SimScore")
              ))
            #.filter(col("SimScore") == 3.0))

      # Show all scores for Bio students
      #exploded_df.filter(col("MajorA") == "Bio").show(truncate=False)

      # Show any pairs with score >= 3
      #exploded_df.filter(col("SimScore") >= 3.0).show(truncate=False)

      # Join back to get StudentB's major
      student_major_df = result_df.select(col("Name").alias("NameB"), col("Major").alias("MajorB"))

      all_pairs_df = (exploded_df.join(student_major_df, exploded_df["StudentB"] == student_major_df["NameB"], "inner")
              .filter(col("MajorA") == col("MajorB"))
              .select(col("StudentA"),col("StudentB"),col("MajorA").alias("Major"),col("SimScore"))
      )


      w = Window.partitionBy("StudentA").orderBy(F.desc("SimScore"))
      best_pairs_df = (all_pairs_df.withColumn("rank", F.row_number().over(w)).filter(col("rank") == 1).drop("rank"))

      best_pairs_df = best_pairs_df.filter(col("SimScore") >= 3.0)

      plagiarism_df = (
        best_pairs_df.withColumn("pair_key",F.when(col("StudentA") < col("StudentB"),
                    F.concat(col("StudentA"), F.lit("-"), col("StudentB")))
                    .otherwise(F.concat(col("StudentB"), F.lit("-"), col("StudentA"))))
                    .dropDuplicates(["pair_key"])
                    .drop("pair_key")
      )


      q = question.lower().strip().rstrip("?").strip()
      # Check if it's a "how many / count" question first

      if any(kw in q for kw in ["how many", "count", "each major"]):
          count_df = plagiarism_df.groupBy("Major").agg(
              F.countDistinct("StudentA").alias("Plagiarism_Count")
          ).orderBy("Major")
          return count_df

      # Otherwise, try to extract a major name for "who plagiarized in X?" style questions
      # Build a list of known majors from the data to help matching
      all_majors = [row["Major"].lower() for row in plagiarism_df.select("Major").distinct().collect()]

      major_input = None

      # Pattern 1: "in major X" or "in the major X"
      if "major" in q:
        words = q.split()
        for i, word in enumerate(words):
            if word == "major" and i + 1 < len(words):
                major_input = words[i + 1]
                break

      # Pattern 2: "in X" at the end (e.g., "who plagiarized in cs", "who cheated in bio")
      if not major_input:
          words = q.split()
          for i, word in enumerate(words):
              if word in ["in", "for"] and i + 1 < len(words):
                  major_input = words[-1]  # take the last word
                  break

      # Pattern 3: major name appears anywhere in the question (fallback)
      if not major_input:
          for maj in all_majors:
            if maj in q:
                major_input = maj
                break

      # If we found a major and the question is about plagiarism/cheating/copying
      plagiarism_keywords = ["plagiarized", "plagiarism", "copied", "cheated", "cheat",
                            "copy", "who", "list", "find", "show", "identify"]

      if major_input and any(kw in q for kw in plagiarism_keywords):
          filtered_df = plagiarism_df.filter(
              F.lower(col("Major")) == major_input.lower()
          ).select("StudentA", "StudentB", "Major", "SimScore")

          if filtered_df.count() == 0:
              print(f"No plagiarism found in Major '{major_input.upper()}'.")
              return spark.createDataFrame([], filtered_df.schema)

          return filtered_df

      # Unknown question
      else:
          print("Sorry, I could not understand the question.")
          print("Try: 'Who plagiarized in Major Bio?' or 'How many students plagiarized in each major?'")
          return spark.createDataFrame([], plagiarism_df.schema)

#question = "Who plagiarized in the Major bio"
#how many students plagiarized in each major?
#Who plagiarized in the Major his?
question = input("Enter question: ")
print("Answer:")
answer_df = answer_question(question, df)
answer_df.show()

Enter question: Who plagiarized in the Major cs??
Answer:
+--------+--------+-----+--------+
|StudentA|StudentB|Major|SimScore|
+--------+--------+-----+--------+
|   Alice|  Daniel|   CS|     3.0|
+--------+--------+-----+--------+



d)	Determine Copying Relationships Based on GPA (20%)
- If the GPA difference ≤ 0.3, classify the relationship as "copy each other".
- If the GPA difference > 0.3, classify as a "copee-copier" relationship:
    - The student with the higher GPA is the copee.
    - The student with the lower GPA is the copier.


In [ ]:
#import Pyspark modules

# Step 2.4.1: Prepare the similarity list
# Start from the deduplicated plagiarism pairs (from last part  logic)

exploded_df = (
    result_df.select(col("Name").alias("StudentA"),col("Major").alias("MajorA"),
        F.explode(F.arrays_zip(col("sim_names"), col("sim_scores"))).alias("pair")
        ).select(col("StudentA"),col("MajorA"),col("pair.sim_names").alias("StudentB"),
        col("pair.sim_scores").alias("SimScore"))
)

student_major_df = result_df.select(col("Name").alias("NameB"), col("Major").alias("MajorB"))

# Step 2.4.2: List out pairs within same major
all_pairs_df = (
    exploded_df
    .join(student_major_df, exploded_df["StudentB"] == student_major_df["NameB"], "inner")
    .filter(col("MajorA") == col("MajorB"))
    .select(col("StudentA"), col("StudentB"), col("MajorA").alias("Major"), col("SimScore"))
)

# Keep only best match per student, filter SimScore >= 3.0, deduplicate
from pyspark.sql.window import Window
w = Window.partitionBy("StudentA").orderBy(F.desc("SimScore"))
best_pairs_df = (
    all_pairs_df.withColumn("rank", F.row_number().over(w)).filter(col("rank") == 1)
    .drop("rank").filter(col("SimScore") >= 3.0)
)

plagiarism_df = (
    best_pairs_df
    .withColumn("pair_key",
        F.when(col("StudentA") < col("StudentB"),F.concat(col("StudentA"), F.lit("-"), col("StudentB")))
        .otherwise(F.concat(col("StudentB"), F.lit("-"), col("StudentA")))
    )
    .dropDuplicates(["pair_key"])
    .drop("pair_key")
)

# Step 2.4.3: Prepare GPA info

# Join GPA for StudentA
gpa_a_df = result_df.select(col("Name").alias("NameA"), col("Past GPA").cast("float").alias("GPA_A"))
# Join GPA for StudentB
gpa_b_df = result_df.select(col("Name").alias("NameB2"), col("Past GPA").cast("float").alias("GPA_B"))

relation_df = (
    plagiarism_df
    .join(gpa_a_df, plagiarism_df["StudentA"] == gpa_a_df["NameA"], "inner")
    .join(gpa_b_df, plagiarism_df["StudentB"] == gpa_b_df["NameB2"], "inner")
    .drop("NameA", "NameB2")
)

# Step 2.4.4: Compare GPA and define relationship type
relation_df = relation_df.withColumn(
    "Relation",
    F.when(F.abs(col("GPA_A") - col("GPA_B")) <= 0.3, "copy each other")
     .otherwise(
         F.when(col("GPA_A") > col("GPA_B"), "copee-copyer")
          .otherwise("copyer-copee")
     )
)

# Step 2.4.5: Show results
relation_df = relation_df.select(
    "StudentA", "GPA_A", "StudentB", "GPA_B", "Major", "SimScore", "Relation"
)

relation_df.show(truncate=False)

+--------+-----+--------+-----+-----+--------+---------------+
|StudentA|GPA_A|StudentB|GPA_B|Major|SimScore|Relation       |
+--------+-----+--------+-----+-----+--------+---------------+
|Alice   |3.5  |Daniel  |3.4  |CS   |3.0     |copy each other|
|Carla   |3.9  |Grace   |1.9  |Bio  |3.0     |copee-copyer   |
+--------+-----+--------+-----+-----+--------+---------------+



**You've reached the end of the assignment. Don't forget to upload it to Blackboard.**